# 04. Geometry for Head Direction (Yaw & Pitch)

**Goal:** Use math to detect if a face is looking straight or turning away.


## 1. The Strategy: Ratios (Heuristics)
True 3D pose estimation is hard. We use a simpler 2D method: **Ratios**.
- If the Nose is exactly between the cheeks, you are facing forward.
- If the Nose is closer to the left cheek, you are looking left.


## 2. Yaw (Left/Right)
`Yaw Ratio = (Nose - LeftCheek) / (RightCheek - LeftCheek)`
- **0.5**: Centered.
- **< 0.4**: Looking Left.
- **> 0.6**: Looking Right.


## 3. Pitch (Up/Down)
`Pitch Ratio = (Nose - Chin) / (Forehead - Chin)`
- **0.5**: Centered.
- **< 0.4**: Looking Down.
- **> 0.6**: Looking Up.


In [ ]:
import cv2
import mediapipe as mp

mp_face_mesh = mp.solutions.face_mesh
face_mesh = mp_face_mesh.FaceMesh(max_num_faces=1, refine_landmarks=True)
cap = cv2.VideoCapture(0)

print("Move your head! 'q' to quit")

while True:
    ret, frame = cap.read()
    if not ret: break
    rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    results = face_mesh.process(rgb)
    h, w, c = frame.shape
    
    if results.multi_face_landmarks:
        for face in results.multi_face_landmarks:
            # Keypoints
            nose = face.landmark[1]
            left_cheek = face.landmark[234]
            right_cheek = face.landmark[454]
            chin = face.landmark[152]
            forehead = face.landmark[10]
            
            # Calc Yaw (Horizontal)
            face_width = abs(right_cheek.x - left_cheek.x)
            if face_width > 0:
                yaw = abs(nose.x - left_cheek.x) / face_width
            else: yaw = 0.5
            
            # Calc Pitch (Vertical)
            face_height = abs(chin.y - forehead.y)
            if face_height > 0:
                pitch = abs(chin.y - nose.y) / face_height
            else: pitch = 0.5
            
            # Draw it
            nx, ny = int(nose.x * w), int(nose.y * h)
            cv2.putText(frame, f"Yaw: {yaw:.2f}", (nx, ny-20), cv2.FONT_HERSHEY_PLAIN, 1, (0,255,0), 1)
            cv2.putText(frame, f"Pitch: {pitch:.2f}", (nx, ny+20), cv2.FONT_HERSHEY_PLAIN, 1, (0,255,255), 1)

    cv2.imshow('Lesson 04', frame)
    if cv2.waitKey(1) & 0xFF == ord('q'): break

cap.release()
cv2.destroyAllWindows()
